# Sprint 0 — Step 1: Indicator & coverage verification

**Verified against the live World Bank API on 2026-06-26** 

Coverage checked by *aligned data windows* (first–last year per country), not just year-counts.

## Countries (frozen): 6 Western Balkans

ALB, BIH, MNE, MKD, SRB — full series **~2000–2024**.
**XKX (Kosovo)** — clean *late start* **2008–2024** on economic/labour (national accounts begin ~2008),
**2000–2024** on governance. Kept in the set; gaps documented per-indicator, not by dropping the country.

## Frozen basket (11 indicators)

| Dimension | Indicator | Code | DB | Coverage |
|---|---|---|---|---|
| Income | GDP per capita, PPP (const **2021** intl $) | `NY.GDP.PCAP.PP.KD` | 2 | all six |
| Investment | Gross fixed capital formation (% GDP) | `NE.GDI.FTOT.ZS` | 2 | all six |
| Investment | FDI net inflows (% GDP) | `BX.KLT.DINV.WD.GD.ZS` | 2 | all six |
| Productivity | GDP per person employed (const 2021 PPP $) | `SL.GDP.PCAP.EM.KD` | 2 | **5 of 6 — Kosovo NONE** |
| Trade | Trade (% GDP) | `NE.TRD.GNFS.ZS` | 2 | all six |
| Labour | Labour force participation (**national est.**) | `SL.TLF.CACT.NE.ZS` | 2 | all six |
| Labour | Unemployment, total (**national est.**) | `SL.UEM.TOTL.NE.ZS` | 2 | all six |
| Labour | Youth unemployment (**national est.**) | `SL.UEM.1524.NE.ZS` | 2 | all six |
| Governance | Rule of Law — score 0–100 | `GOV_WGI_RL.SC` | 3 | all six |
| Governance | Control of Corruption — score 0–100 | `GOV_WGI_CC.SC` | 3 | all six |
| Governance | Government Effectiveness — score 0–100 | `GOV_WGI_GE.SC` | 3 | all six |

## Key decisions & findings (the "why")

1. **Governance codes changed.** The 2025 WGI revision retired the old `*.PER.RNK` percentile ranks
   (used in the proposal — now dead in the API) and replaced them with `GOV_WGI_*.SC`, an **absolute
   0–100 score**. Methodology note: *absolute score, not percentile rank*. Still not ratio-scaled: 
   governance stays **descriptive, own axis, no gap-to-EU / years-to-close / sigma**.
2. **Governance lives in `source=3`** and `wbgapi` cannot fetch it (JSON error). Governance is pulled via
   **raw `requests`**; the rest of the basket via `wbgapi` (`db=2`).
3. **Labour = national estimates, not modeled ILO.** Modeled-ILO series are **empty for Kosovo (0/15)**;
   national estimates cover it (13/15). Trade-off: national series are slightly less cross-country
   consistent (each office's own methodology) — accepted as the price of keeping Kosovo.
4. **Productivity kept 5-country**, Kosovo documented absent. Excluded from any six-country dispersion
   (sigma) metric; reported for the five.
5. **Innovation dimension dropped — documented finding, not omission.** Probed R&D (`GB.XPD.RSDV.GD.ZS`),
   high-tech exports, researchers/million, patents, IP receipts; plus candidate additions tertiary/secondary
   enrolment, education spend, internet users, mobile subs. **Kosovo is empty or near-empty on every genuine
   innovation/human-capital/digital indicator** — a structural data-infrastructure limitation. The only
   fully-covered option (IP receipts, USD) does not actually measure innovation. Excluded to preserve
   all-six comparability.
6. **Minor window caveat:** Government Effectiveness starts later for MNE and XKX (~2006). Inside the
   analysis window; noted.
7. **PPP series rebased to constant 2021 international $** (was 2017).

## Open — decided in Step 2
**Analysis window choice:** common **2008–2024** (all six, balanced) vs **2000–2024** for five + 2008–2024
for Kosovo (longer, ragged panel — biases sigma in 2000–2007). The recent-trend "stuck" metric (~2014–2024)
is unaffected either way.

In [1]:
import wbgapi as wb

In [2]:
# 1. WDI basket (default db=2): confirm each code resolves + print its official name
wb.series.info(['NY.GDP.PCAP.PP.KD','SL.GDP.PCAP.EM.KD','NE.GDI.FTOT.ZS',
                'BX.KLT.DINV.WD.GD.ZS','NE.TRD.GNFS.ZS','GB.XPD.RSDV.GD.ZS',
                'SL.TLF.CACT.ZS','SL.UEM.TOTL.ZS','SL.UEM.1524.ZS'])

id,value
BX.KLT.DINV.WD.GD.ZS,"Foreign direct investment, net inflows (% of GDP)"
NY.GDP.PCAP.PP.KD,"GDP per capita, PPP (constant 2021 international $)"
SL.GDP.PCAP.EM.KD,GDP per person employed (constant 2021 PPP $)
NE.GDI.FTOT.ZS,Gross fixed capital formation (% of GDP)
SL.TLF.CACT.ZS,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)"
GB.XPD.RSDV.GD.ZS,Research and development expenditure (% of GDP)
NE.TRD.GNFS.ZS,Trade (% of GDP)
SL.UEM.TOTL.ZS,"Unemployment, total (% of total labor force) (modeled ILO estimate)"
SL.UEM.1524.ZS,"Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)"
,9 elements


In [4]:
# 2. Governance lives in source 3 — pull actual DATA for the six, not metadata
wb.db = 3
gov = wb.data.DataFrame(
    ['RL.PER.RNK','CC.PER.RNK','GE.PER.RNK'],
    ['ALB','BIH','XKX','MNE','MKD','SRB'],
    mrv=5  # most recent 5 years
)
wb.db = 2  # switch back to WDI
gov

APIResponseError: APIError: JSON decoding error (https://api.worldbank.org/v2/en/sources/3/series/RL.PER.RNK;CC.PER.RNK;GE.PER.RNK/country/ALB;BIH;XKX;MNE;MKD;SRB/time/all?per_page=1000&mrv=5&page=1&format=json)

In [5]:
import requests
url = "https://api.worldbank.org/v2/country/ALB/indicator/RL.PER.RNK?source=3&format=json&mrv=5"
r = requests.get(url)
print(r.status_code)
print(r.text[:600])

200
[{"message":[{"id":"120","key":"Invalid value","value":"The provided parameter value is not valid"}]}]


In [6]:
import requests
tests = {
  "A v2 singular + PER.RNK": "https://api.worldbank.org/v2/country/alb/indicator/RL.PER.RNK?source=3&format=json&mrv=2",
  "B v2 plural + EST":       "https://api.worldbank.org/v2/countries/alb/indicators/CC.EST?source=3&format=json&mrv=2",
  "C list source-3 series":  "https://api.worldbank.org/v2/source/3/series?format=json&per_page=5",
}
for name, u in tests.items():
    r = requests.get(u)
    print(name, "->", r.status_code, "|", r.text[:160].replace("\n", " "))
    print()

A v2 singular + PER.RNK -> 200 | [{"message":[{"id":"120","key":"Invalid value","value":"The provided parameter value is not valid"}]}]

B v2 plural + EST -> 200 | [{"message":[{"id":"120","key":"Invalid value","value":"The provided parameter value is not valid"}]}]

 <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd



In [7]:
import requests

# 1. Did the 2025 revision fold WGI into the default DB? Test RL.PER.RNK with NO source param.
r1 = requests.get("https://api.worldbank.org/v2/country/alb/indicator/RL.PER.RNK?format=json&mrv=2")
print("no-source ->", r1.status_code, "|", r1.text[:200].replace("\n", " "))
print()

# 2. What sources does the v2 API actually list now? Find WGI's current id (or its absence).
r2 = requests.get("https://api.worldbank.org/v2/sources?format=json&per_page=200")
try:
    for s in r2.json()[1]:
        print(s["id"], "-", s["name"])
except Exception as e:
    print("parse failed:", e, "|", r2.text[:200])

no-source -> 200 | [{"message":[{"id":"175","key":"Invalid format","value":"The indicator was not found. It may have been deleted or archived."}]}]

1 - Doing Business
2 - World Development Indicators
3 - Worldwide Governance Indicators
5 - Subnational Malnutrition Database
6 - International Debt Statistics
11 - Africa Development Indicators
12 - Education Statistics
13 - Enterprise Surveys
14 - Gender Statistics
15 - Global Economic Monitor
16 - Health Nutrition and Population Statistics
18 - IDA Results Measurement System
19 - Millennium Development Goals
20 - Quarterly Public Sector Debt
22 - Quarterly External Debt Statistics SDDS
23 - Quarterly External Debt Statistics GDDS
25 - Jobs
27 - Global Economic Prospects
28 - Global Findex database
29 - The Atlas of Social Protection: Indicators of Resilience and Equity
30 - Exporter Dynamics Database – Indicators at Country-Year Level
31 - Country Policy and Institutional Assessment
32 - Global Financial Development
33 - G20 Financial I

In [8]:
import requests
r = requests.get("https://api.worldbank.org/v2/indicator?source=3&format=json&per_page=200")
data = r.json()
print("status:", r.status_code, "| count:", data[0].get("total"))
for ind in data[1]:
    print(ind["id"], "-", ind["name"])

status: 200 | count: 36
GOV_WGI_CC.EST - Control of Corruption - Governance estimate (approx. -2.5 to +2.5)
GOV_WGI_CC.SC - Control of Corruption - Governance score (0-100)
GOV_WGI_CC.SC_LB - Control of Corruption - Lower bound of the 90% confidence interval for the governance score
GOV_WGI_CC.SC_UB - Control of Corruption - Upper bound of the 90% confidence interval for the governance score
GOV_WGI_CC.SE - Control of Corruption - Standard error of the governance estimate
GOV_WGI_CC.SR - Control of Corruption - Number of sources
GOV_WGI_GE.EST - Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)
GOV_WGI_GE.SC - Government Effectiveness - Governance score (0-100)
GOV_WGI_GE.SC_LB - Government Effectiveness - Lower bound of the 90% confidence interval for the governance score
GOV_WGI_GE.SC_UB - Government Effectiveness - Upper bound of the 90% confidence interval for the governance score
GOV_WGI_GE.SE - Government Effectiveness - Standard error of the governance estima

In [9]:
import requests, pandas as pd

codes = ["GOV_WGI_RL.SC", "GOV_WGI_CC.SC", "GOV_WGI_GE.SC"]   # new 0-100 score
countries = "alb;bih;xkx;mne;mkd;srb"
rows = []
for c in codes:
    url = (f"https://api.worldbank.org/v2/country/{countries}/indicator/{c}"
           f"?source=3&format=json&mrv=5&per_page=500")
    j = requests.get(url).json()
    for d in j[1]:
        rows.append({"indicator": c.replace("GOV_WGI_",""),
                     "country": d["countryiso3code"],
                     "year": d["date"], "value": d["value"]})
pd.DataFrame(rows).pivot_table(index=["country","year"],
                               columns="indicator", values="value")

indicator         CC.SC      GE.SC      RL.SC
country year                                 
ALB     2020  34.506940  51.704282  52.546065
        2021  35.668816  53.511147  53.495745
        2022  37.724830  55.092529  54.710700
        2023  38.461857  57.045918  54.331282
        2024  39.337997  57.413696  54.925199
BIH     2020  35.210761  37.630076  51.756881
        2021  34.313128  35.000461  51.673256
        2022  34.089345  36.835772  51.978064
        2023  35.741963  38.791885  50.856969
        2024  37.473588  38.987376  51.472190
MKD     2020  38.604912  49.545924  53.987636
        2021  40.048872  49.808552  54.949492
        2022  41.121015  51.556397  53.862899
        2023  40.685729  51.775072  52.251033
        2024  41.190881  50.711606  53.093512
MNE     2020  47.126371  53.098399  58.917036
        2021  48.630997  50.861671  57.271254
        2022  47.596841  51.716211  56.024827
        2023  46.476830  56.720567  59.161619
        2024  46.651784  58.466611  60.086444
SRB     2020  40.401332  52.267464  54.643787
        2021  38.325360  52.562483  53.649332
        2022  38.039608  54.817612  54.280608
        2023  37.699812  53.353788  53.797292
        2024  36.502794  53.322769  53.521231
XKX     2020  39.280716  44.115040  52.171931
        2021  39.921832  48.269381  54.830970
        2022  41.881136  50.878249  54.396450
        2023  44.858451  51.520468  56.211689
        2024  42.698736  50.148657  57.862011

In [10]:
# R&D coverage probe + first real wbgapi data pull
wb.db = 2  # force back to WDI (the failed cell may have left it on 3)
rnd = wb.data.DataFrame('GB.XPD.RSDV.GD.ZS',
                        ['ALB','BIH','XKX','MNE','MKD','SRB'], mrv=15)
rnd

,YR2010,YR2011,YR2012,YR2013,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023,YR2024
economy,,,,,,,,,,,,,,,
ALB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.18998,0.19997,NaN,NaN
BIH,NaN,NaN,0.26533,0.32132,0.25725,0.21603,0.21337,0.19780,0.19199,0.19001,0.20325,0.19035,0.18743,0.18853,0.21437
MKD,0.21634,0.22276,0.32676,0.43894,0.51646,0.44412,0.43585,0.35439,0.36367,0.36783,0.37264,0.37252,0.37372,0.36990,0.37363
MNE,NaN,0.31485,NaN,0.37431,0.36320,0.37400,0.32454,0.34898,0.50374,0.36328,NaN,NaN,NaN,NaN,NaN
SRB,0.67458,0.65677,0.81922,0.65574,0.69287,0.77867,0.80779,0.83834,0.88154,0.84798,0.86478,0.94784,0.91938,0.87816,0.93978
XKX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Probe innovation proxies — same NaN-pattern test we ran on R&D
wb.db = 2
proxies = {
    'TX.VAL.TECH.MF.ZS': 'HighTech exports %',
    'SP.POP.SCIE.RD.P6': 'Researchers/million',
    'IP.PAT.RESD':       'Patents (residents)',
    'BX.GSR.ROYL.CD':    'IP receipts $',
}
for code, label in proxies.items():
    df = wb.data.DataFrame(code, ['ALB','BIH','XKX','MNE','MKD','SRB'], mrv=15)
    coverage = df.notna().sum(axis=1)   # non-null years per country
    print(f"\n=== {label}  ({code}) ===")
    print(coverage.to_string())


=== HighTech exports %  (TX.VAL.TECH.MF.ZS) ===
economy
ALB    15
BIH    15
MKD    15
MNE    15
SRB     0
XKX     0

=== Researchers/million  (SP.POP.SCIE.RD.P6) ===
economy
ALB     2
BIH    13
MKD    15
MNE     8
SRB    15
XKX     0

=== Patents (residents)  (IP.PAT.RESD) ===
economy
ALB     8
BIH    12
MKD     9
MNE    11
SRB    15
XKX     0

=== IP receipts $  (BX.GSR.ROYL.CD) ===
economy
ALB    15
BIH    15
MKD    15
MNE    15
SRB    15
XKX    15


In [12]:
# Probe candidate ADD-ON dimensions — human capital + digital — Kosovo is the gatekeeper
wb.db = 2
candidates = {
    'SE.TER.ENRR':    'Tertiary enrolment (gross %)',     # human capital
    'SE.SEC.ENRR':    'Secondary enrolment (gross %)',     # human capital
    'SE.XPD.TOTL.GD.ZS':'Govt education spend (% GDP)',     # human capital
    'IT.NET.USER.ZS': 'Internet users (% pop)',            # digital
    'IT.CEL.SETS.P2': 'Mobile subs (per 100)',             # digital
}
for code, label in candidates.items():
    df = wb.data.DataFrame(code, ['ALB','BIH','XKX','MNE','MKD','SRB'], mrv=15)
    coverage = df.notna().sum(axis=1)
    print(f"\n=== {label}  ({code}) ===")
    print(coverage.to_string())


=== Tertiary enrolment (gross %)  (SE.TER.ENRR) ===
economy
ALB    15
BIH    14
MKD    12
MNE    14
SRB    15
XKX     0

=== Secondary enrolment (gross %)  (SE.SEC.ENRR) ===
economy
ALB    15
BIH    14
MKD    12
MNE    12
SRB    14
XKX     0

=== Govt education spend (% GDP)  (SE.XPD.TOTL.GD.ZS) ===
economy
ALB    14
BIH    10
MKD     0
MNE     0
SRB    10
XKX     0

=== Internet users (% pop)  (IT.NET.USER.ZS) ===
economy
ALB    14
BIH    14
MKD    15
MNE    14
SRB    14
XKX     2

=== Mobile subs (per 100)  (IT.CEL.SETS.P2) ===
economy
ALB    15
BIH    15
MKD    15
MNE    15
SRB    15
XKX     2


In [13]:
# Validate the FROZEN basket: do the 8 core indicators cover all six? Kosovo is the test.
# Plus the NATIONAL labour variants, to settle the modeled-vs-national decision with data.
wb.db = 2
import pandas as pd
frozen = {
    'NY.GDP.PCAP.PP.KD':   'GDP/cap PPP',
    'SL.GDP.PCAP.EM.KD':   'Productivity',
    'NE.GDI.FTOT.ZS':      'GFCF %GDP',
    'BX.KLT.DINV.WD.GD.ZS':'FDI %GDP',
    'NE.TRD.GNFS.ZS':      'Trade %GDP',
    'SL.TLF.CACT.ZS':      'LFP (modeled)',
    'SL.UEM.TOTL.ZS':      'Unemp (modeled)',
    'SL.UEM.1524.ZS':      'Youth unemp (modeled)',
    'SL.TLF.CACT.NE.ZS':   'LFP (national)',     # comparison for the decision
    'SL.UEM.TOTL.NE.ZS':   'Unemp (national)',   # comparison for the decision
}
out = {}
for code, label in frozen.items():
    df = wb.data.DataFrame(code, ['ALB','BIH','XKX','MNE','MKD','SRB'], mrv=15)
    out[label] = df.notna().sum(axis=1)
pd.DataFrame(out).T

economy,ALB,BIH,MKD,MNE,SRB,XKX
GDP/cap PPP,15,15,15,15,15,15
Productivity,15,15,15,15,15,0
GFCF %GDP,15,15,15,15,15,15
FDI %GDP,15,15,15,15,15,15
Trade %GDP,15,15,15,15,15,15
LFP (modeled),15,15,15,15,15,0
Unemp (modeled),15,15,15,15,15,0
Youth unemp (modeled),15,15,15,15,15,0
LFP (national),15,15,15,15,15,13
Unemp (national),15,15,15,15,15,13


In [14]:
wb.db = 2
import pandas as pd
labour_natl = {
    'SL.TLF.CACT.NE.ZS':  'LFP (national)',
    'SL.UEM.TOTL.NE.ZS':  'Unemp (national)',
    'SL.UEM.1524.NE.ZS':  'Youth unemp (national)',  # untested
}
out = {}
for code, label in labour_natl.items():
    try:
        df = wb.data.DataFrame(code, ['ALB','BIH','XKX','MNE','MKD','SRB'], mrv=15)
        out[label] = df.notna().sum(axis=1)
    except Exception as e:
        out[label] = pd.Series({'ERROR': str(e)[:50]})
pd.DataFrame(out).T

economy,ALB,BIH,MKD,MNE,SRB,XKX
LFP (national),15,15,15,15,15,13
Unemp (national),15,15,15,15,15,13
Youth unemp (national),15,15,15,14,15,13


In [15]:
import pandas as pd
countries = ['ALB','BIH','XKX','MNE','MKD','SRB']

# final frozen basket — WDI codes (db=2) and governance (db=3) handled separately
wdi = ['NY.GDP.PCAP.PP.KD','NE.GDI.FTOT.ZS','BX.KLT.DINV.WD.GD.ZS',
       'SL.GDP.PCAP.EM.KD','NE.TRD.GNFS.ZS',
       'SL.TLF.CACT.NE.ZS','SL.UEM.TOTL.NE.ZS','SL.UEM.1524.NE.ZS']

def span(df):
    # for each country (row), first & last year with a non-null value
    out = {}
    for c in df.index:
        yrs = [int(col[2:]) for col in df.columns if pd.notna(df.loc[c, col])]
        out[c] = f"{min(yrs)}–{max(yrs)} ({len(yrs)})" if yrs else "NONE"
    return out

rows = {}
wb.db = 2
for code in wdi:
    rows[code] = span(wb.data.DataFrame(code, countries, mrv=25))

print(pd.DataFrame(rows).T.to_string())

                                 ALB             BIH             MKD             MNE             SRB             XKX
NY.GDP.PCAP.PP.KD     2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2008–2024 (17)
NE.GDI.FTOT.ZS        2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2008–2024 (17)
BX.KLT.DINV.WD.GD.ZS  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2007–2024 (18)  2000–2024 (25)  2008–2024 (17)
SL.GDP.PCAP.EM.KD     2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)            NONE
NE.TRD.GNFS.ZS        2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2000–2024 (25)  2008–2024 (17)
SL.TLF.CACT.NE.ZS     2000–2024 (24)  2001–2024 (20)  2000–2024 (24)  2007–2024 (18)  2004–2024 (21)  2000–2024 (22)
SL.UEM.TOTL.NE.ZS     2000–2024 (25)  2001–2024 (20)  2000–2024 (25)  2005–2024 (19)  2000–2024 (25)  2000–2024 (23)
SL.UEM.1524.NE.ZS     2002–2024 (20)  2001–2024 (20)  2002–2024 

In [16]:
wb.db = 3
import requests, pandas as pd
gov = {'GOV_WGI_RL.SC':'RL','GOV_WGI_CC.SC':'CC','GOV_WGI_GE.SC':'GE'}
rows = {}
for code,label in gov.items():
    url=(f"https://api.worldbank.org/v2/country/alb;bih;xkx;mne;mkd;srb/"
         f"indicator/{code}?source=3&format=json&mrv=30&per_page=2000")
    j=requests.get(url).json()
    span={}
    for d in j[1]:
        if d["value"] is not None:
            c=d["countryiso3code"]; y=int(d["date"])
            span.setdefault(c,[]).append(y)
    rows[label]={c:f"{min(v)}–{max(v)} ({len(v)})" for c,v in span.items()}
wb.db=2
print(pd.DataFrame(rows).T.to_string())

               ALB             BIH             MKD             MNE             SRB             XKX
RL  1996–2024 (26)  1996–2024 (26)  1996–2024 (26)  1998–2024 (25)  1996–2024 (26)  2000–2024 (24)
CC  1996–2024 (26)  1996–2024 (26)  1996–2024 (26)  1998–2024 (25)  1996–2024 (26)  2000–2024 (24)
GE  1996–2024 (26)  1996–2024 (26)  1996–2024 (26)  2006–2024 (19)  1996–2024 (26)  2006–2024 (19)


# Sprint 0 — Step 2: Analysis time-window decision

**Decision: common window 2008–2024 (16 years), all six countries aligned.**

## Why
- **Kosovo (XKX) economic/labour data begins 2008** (national accounts start ~2008); the other five
  run from ~2000. A common window removes the ragged panel.
- **Thesis focus is recent catch-up (roughly last 10–15 years)**, not the early 2000s — so 2000–2007 data
  for the five longer series adds nothing the analysis uses.
- **Sigma metric.** On a ragged 2000–2024 panel, dispersion would be computed on
  5 countries (2000–2007) then 6 (2008+); that seam can masquerade as convergence/divergence that is
  really just Kosovo entering the sample. The common window eliminates this.

## What runs on this window
- **All three convergence metrics** (gap-to-EU, years-to-close, sigma): **2008–2024, six countries.**
- **"Stuck" trend** (slope over most recent 10 yrs, ≈2014–2024): sits inside the window. Unaffected.
- **Productivity:** five-country (Kosovo NONE) on 2008–2024.
- **Governance:** own axis / descriptive; data exists back to 1996/2000 but shown on the same 2008–2024
  frame for consistency (revisit when building the governance panel again).

## NOTE:
The window opens on the **2008–2009 global financial crisis**. Movement in the first 1–2 years partly
reflects differential GFC impact, not structural convergence. The recent-trend focus (2014+) sidesteps
this; flagged so early-window movement isn't over-interpreted as a structural trend.